# 13 — SAM 3 nativo desde HuggingFace *(Avanzado / Opcional)*

> ⚠️ **Este notebook es avanzado y opcional.**
> Requiere haber completado NB06 y NB07, tener acceso aprobado al modelo
> en HuggingFace y una sesión autenticada. Los notebooks del curso no dependen de este.

## ¿Qué vamos a construir hoy?

Usarás la API nativa de HuggingFace (`transformers`) para cargar SAM 3
directamente, sin el wrapper de Ultralytics. Luego convertirás el output
al formato de Supervision — demostrando que la librería funciona con
cualquier API, no solo con Ultralytics.

**Aprenderás a:**
- Cargar SAM 3 con `Sam3Processor` y `Sam3Model` de `transformers`
- Correr inferencia con prompts de texto y bounding boxes
- Convertir el output nativo a `sv.Detections` manualmente
- Entender qué abstrae Ultralytics y qué ganas con la API nativa

**Prerequisitos:**
- Cuenta en HuggingFace con acceso aprobado a `facebook/sam3`
- Token de acceso generado en https://huggingface.co/settings/tokens

**Tiempo estimado:** 45 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## API nativa vs. wrapper de Ultralytics

En NB06 usamos `SAM("sam3.pt")` de Ultralytics — conveniente pero limitado
a lo que el wrapper expone.

La API nativa de HuggingFace da acceso a:
- Todos los tipos de prompt: texto, cajas, puntos, máscaras, ejemplares visuales
- Control fino sobre umbrales de segmentación
- Inferencia en lote (batch) sobre múltiples imágenes
- Pesos del modelo sin conversión

```
Ultralytics:     SAM("sam3.pt")(image, bboxes=...)  ← simple
HuggingFace:     processor(images, text) → model() → post_process()  ← flexible
```

El costo: más código explícito. El beneficio: acceso completo al modelo.

Supervision sigue siendo el puente al final — el `sv.Detections` que
construyamos aquí es idéntico al de cualquier otro notebook.


In [ ]:
!pip install transformers torch supervision
!huggingface-cli login   # ejecuta esto una vez para autenticarte
# !hf auth login

import supervision as sv
from transformers import Sam3Processor, Sam3Model
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import urllib.request
from pathlib import Path
from PIL import Image

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "assets/bus.jpg")
urllib.request.urlretrieve("https://ultralytics.com/images/zidane.jpg", "assets/zidane.jpg")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
# En CPU SAM 3 es lento (~30–60 s por imagen); en GPU es fluido.


In [ ]:
from huggingface_hub import login
login(token="token")

## Paso 1: Autenticarse y cargar el modelo

Si aún no has iniciado sesión en HuggingFace, ejecuta en la terminal:

```bash
huggingface-cli login
```

Pega tu token de acceso cuando se solicite. Solo necesitas hacerlo una vez por entorno.

Asegúrate también de haber solicitado y recibido acceso al modelo en:
https://huggingface.co/facebook/sam3


In [ ]:
# El processor convierte imágenes y prompts al formato que el modelo espera
# El model es la red neuronal que genera las máscaras
processor = Sam3Processor.from_pretrained("facebook/sam3")
model     = Sam3Model.from_pretrained("facebook/sam3").to(device)
model.eval()   # desactivar dropout y otras capas de entrenamiento

print("Modelo cargado")
print(f"Parámetros: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} M")


## Paso 2: Inferencia con prompt de texto

La API de `transformers` tiene tres pasos:

1. **`processor()`** — prepara imagen y prompt como tensores
2. **`model()`** — ejecuta la red neuronal
3. **`post_process_instance_segmentation()`** — convierte el output bruto a máscaras y cajas

El resultado es un diccionario con `masks`, `boxes` y `scores`.
Necesitamos convertirlo a `sv.Detections` manualmente.


In [ ]:
# Función auxiliar: convierte el output de transformers a sv.Detections
def sam3_a_detections(results: dict) -> sv.Detections:
    masks  = results["masks"].cpu().numpy().astype(bool)   # (N, H, W)
    xyxy   = results["boxes"].cpu().numpy()                 # (N, 4) formato x1y1x2y2
    scores = results["scores"].cpu().numpy()                # (N,)
    return sv.Detections(xyxy=xyxy, mask=masks, confidence=scores)


# Cargar imagen como PIL (transformers usa PIL, OpenCV usa BGR)
image_pil = Image.open("assets/bus.jpg").convert("RGB")
image_bgr = cv2.imread("assets/bus.jpg")   # solo para anotar con Supervision

# Preparar inputs: imagen + texto
inputs = processor(
    images=image_pil,
    text="person",
    return_tensors="pt"
).to(device)

# Inferencia sin gradientes (más rápido, menos memoria)
with torch.no_grad():
    outputs = model(**inputs)

# Post-procesado: convierte tensores brutos a máscaras binarias y cajas
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=[image_pil.size[::-1]]   # (alto, ancho)
)[0]

detections = sam3_a_detections(results)

print(f"Objetos encontrados: {len(detections)}")
print(f"¿Tiene máscaras?    {detections.mask is not None}")
if detections.mask is not None:
    print(f"Shape de máscaras:  {detections.mask.shape}")


## Paso 3: Visualizar con Supervision

Una vez convertido a `sv.Detections`, el resto es idéntico a NB06.
MaskAnnotator no sabe ni le importa de qué API vienen las máscaras.


In [ ]:
mask_annotator  = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX)
label_annotator = sv.LabelAnnotator(text_scale=0.5, color_lookup=sv.ColorLookup.INDEX)

# Etiquetas: confianza de cada máscara
labels = [f"{c:.2f}" for c in detections.confidence]

annotated = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("SAM 3 nativo (transformers) → sv.Detections → MaskAnnotator")
plt.show()


## Pausa y observa: estructura del output nativo

Antes de `sam3_a_detections()`, el output tiene este formato:


In [ ]:
print("Claves del output post-procesado:", list(results.keys()))
print()
print(f"masks  → tipo: {type(results['masks'])},  shape: {results['masks'].shape}")
print(f"boxes  → tipo: {type(results['boxes'])},  shape: {results['boxes'].shape}")
print(f"scores → tipo: {type(results['scores'])}, shape: {results['scores'].shape}")
print()
print("Después de sam3_a_detections():")
print(f"  xyxy:       {detections.xyxy.shape}")
print(f"  mask:       {detections.mask.shape}")
print(f"  confidence: {detections.confidence}")
# La conversión es: tensor GPU → numpy CPU → sv.Detections
# Eso es todo lo que necesita Supervision para trabajar con el resultado.


## 🔧 Exploración interactiva

### Experimento 1: Múltiples conceptos en un prompt

SAM 3 puede segmentar varios conceptos en una sola llamada.


In [ ]:
# Segmentar 'person' y 'bus' en la misma imagen
inputs_multi = processor(
    images=image_pil,
    text=[["person", "bus"]],   # lista de conceptos
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs_multi = model(**inputs_multi)

results_multi = processor.post_process_instance_segmentation(
    outputs_multi,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=[image_pil.size[::-1]]
)[0]

det_multi = sam3_a_detections(results_multi)
print(f"Objetos con ['person', 'bus']: {len(det_multi)}")

annotated_multi = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(
    scene=image_bgr.copy(), detections=det_multi
)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated_multi, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Prompt: ['person', 'bus']")
plt.show()
# 💭 Reflexión: ¿El modelo segmenta los dos conceptos por separado o juntos?
# Juntos — devuelve todas las instancias de ambos conceptos en el mismo resultado.


### Experimento 2: Texto vs. bounding box — ¿el mismo resultado?

Compara usar texto ("person") contra usar las cajas de YOLO.
¿Las máscaras son equivalentes?


In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Obtener cajas con YOLO
yolo_model  = YOLO("yolov8n.pt")
yolo_r      = yolo_model(image_bgr)[0]
yolo_det    = sv.Detections.from_ultralytics(yolo_r)
solo_person = yolo_det[yolo_det.class_id == 0]   # clase 0 = person

# SAM 3 con bounding boxes de YOLO
boxes_list = solo_person.xyxy.tolist()
inputs_bbox = processor(
    images=image_pil,
    # Envolvemos TODA la lista en un solo corchete extra -> 1 imagen con N cajas
    input_boxes=[boxes_list],     
    # Igual aquí: 1 lista con N etiquetas "1" adentro
    input_boxes_labels=[[1] * len(boxes_list)],
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs_bbox = model(**inputs_bbox)

results_bbox = processor.post_process_instance_segmentation(
    outputs_bbox,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=[image_pil.size[::-1]]
)[0]
det_bbox = sam3_a_detections(results_bbox)

# Comparar lado a lado
scene_txt  = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(scene=image_bgr.copy(), detections=detections)
scene_bbox = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(scene=image_bgr.copy(), detections=det_bbox)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
ax1.imshow(cv2.cvtColor(scene_txt,  cv2.COLOR_BGR2RGB)); ax1.set_title('Prompt texto: "person"');  ax1.axis("off")
ax2.imshow(cv2.cvtColor(scene_bbox, cv2.COLOR_BGR2RGB)); ax2.set_title("Prompt bbox (YOLO)");      ax2.axis("off")
plt.suptitle("Texto vs. bounding box — ¿producen las mismas máscaras?", fontsize=13)
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿Cuándo preferirías uno sobre el otro?
# Texto: no necesitas un detector previo, útil cuando empiezas desde cero.
# Bbox: más preciso cuando ya tienes un detector entrenado para tu dominio.


### Experimento 3: Efecto del umbral de confianza

`threshold` controla cuándo una detección se incluye en el resultado.
Un umbral bajo devuelve más objetos (incluidos falsos positivos);
uno alto, solo los más seguros.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, thr in zip(axes, [0.2, 0.5, 0.8]):
    res_thr = processor.post_process_instance_segmentation(
        outputs,
        threshold=thr,
        mask_threshold=0.5,
        target_sizes=[image_pil.size[::-1]]
    )[0]
    det_thr = sam3_a_detections(res_thr)
    scene   = sv.MaskAnnotator(opacity=0.6, color_lookup=sv.ColorLookup.INDEX).annotate(scene=image_bgr.copy(), detections=det_thr)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(f"threshold={thr}  ({len(det_thr)} objetos)")
    ax.axis("off")

plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿Qué threshold usarías en producción?
# Depende del caso: alta precisión (manufactura) → umbral alto.
# Alta cobertura (seguridad) → umbral bajo y filtrado posterior.


## 🚀 Reto de extensión

**Tarea:** Encapsula todo el flujo en una función `segmentar_con_texto()`
que reciba una ruta de imagen y un texto, y devuelva un `sv.Detections`.
Debe funcionar con cualquier imagen y cualquier concepto.

**Firma esperada:**
```python
def segmentar_con_texto(ruta_imagen: str, texto: str, umbral: float = 0.5) -> sv.Detections:
    ...
```

**Pista:**
```python
# Dentro de la función:
image_pil = Image.open(ruta_imagen).convert("RGB")
inputs    = processor(images=image_pil, text=texto, return_tensors="pt").to(device)
# ... modelo, post_process, sam3_a_detections ...
```

Pruébala con `zidane.jpg` y el texto `"face"`.


In [ ]:
# Escribe tu solución aquí
def segmentar_con_texto(ruta_imagen: str, texto: str, umbral: float = 0.5) -> sv.Detections:
    image_pil = Image.open(ruta_imagen).convert("RGB")
    # inputs = ...
    # outputs = ...
    # results = ...
    # return sam3_a_detections(results)
    pass


# det = segmentar_con_texto("assets/zidane.jpg", "face")
# print(f"Rostros encontrados: {len(det)}")
